# Abstract

The llm-workbench python library is a toolkit designed to guide data scientists and machine learning practitioners through the process of evaluating multiple LLMs for performance on custom tasks, including agentic tasks. While some libraries already exist to assist with certain parts of this kind of workflow, this library combines the necessary stages into a single utility, reducing the overhead for users, and also reducing the risk of omitting a key step required to achieve a rigorous result. 

Evaluating LLMs along multiple metrics, instead of choosing a single evaluation alone, is vital because performance of LLMs is highly variable along multiple fronts. In addition, conducting evaluations with statistical power is key because the nondeterministic nature of LLM behavior can lead to wide swings in performance from case to case. This library guides users in creating evals that cover all the attributes of LLM performance that are relevant to their use case, as well as creating an experiment large enough to achieve statistical significance, including providing easy-to-use analysis reports illustrating that significance and the pertinent outcomes. 

# Links

* https://github.com/Nebulock-Inc/llm-workbench
* License []
* Docs
* Package Index

# Introduction



The problem: why evaluating LLM applications is hard (non-determinism, prompt drift, no ground truth, org/data isolation, reproducibility)
The gap in current tooling / why existing options fall short
What llm-lab is, in one paragraph
Contributions / what this paper covers

# Background & Motivation

The "evals-as-code" philosophy and why it matters
Target users and use cases (e.g., judge-based evaluation, regression testing, model comparison)
Design goals and non-goals

# Core Concepts

The workflow in this library centers around paired-prompt analyses, so the user should frame the experiments around each LLM being tested responding to the same prompts. This enables the statistical analyses that are produced at the end. 



---
Terminology and mental model (evals, judges, datasets, runs, metrics)
How the pieces fit together



# Architecture & Design

This library runs every candidate model over every prompt in a dataset, scores the
outputs with configurable evals, and applies a paired-statistics battery to
declare a winner. Everything is driven through one CLI (`llm-workbench`).

## System Diagram

![](architecture-overview.png)

## Five Layers

1. **CLI** (`cli/`) — seven command groups over a Click app; Rich output helpers.
2. **Orchestration** (`runner/`) — `run_comparison()` sweeps model × record,
   fail-fast on bad model ids, crash-safe/resumable writes to `results.jsonl`.
3. **Three pillars run per row:**
   - **Tasks** (`tasks/`) — produce a `TaskOutput` (the model-under-test call),
     or `ReplayTask` scores a pre-existing answer.
   - **Models** (`models/`) — a prefix registry fans `make_client()` out to six
     backends; SDKs are lazy-imported so the base install stays lean.
   - **Evals + Scoring** (`evals/`, `scoring/`) — an `EvalSpec` drives an LLM
     judge, deterministic programmatic checks, and ground-truth overrides.
4. **Results store** — `results.jsonl` → `results.csv`, deduped to latest per
   (model, record).
5. **Analysis** (`analysis/`, behind the `analysis` extra) — long-format frames →
   a paired/repeated-measures stats battery → `analysis.json/.md`, plots, power.

A shared spine (`object_types.py`, `config.py`) carries provider-agnostic value
types and env config through every layer. Downstream app repos
(`ai-checker-evals`, `context-graph-evals`) depend on llm-workbench as a git dependency
and reuse only a thin public surface (`fill_template`, `GenerationRequest`,
`TaskOutput`, `Task`), shipping their own custom `Task`, evals, and datasets.


## Key Components

**CLI**

The CLI is the primary user surface. There are seven command categories included in the CLI (see diagram) which enable interacting with the different stages of the workflow.

**Datasets**

Users will either use the CLI or manually create a case dataset to use the framework. This should be jsonl data, and each line's json object includes all the input data that the LLM would receive that is specific to a single case in a live call in the production setting. This does not include the overall prompt instructions ("You are an analyst, your output format should be XYZ") but should just be the specific details that differentiate each case from one another.

**Evals and Specs**

Evals are declared definitions of the aspects of assessment you wish to cover in your experiment. Examples may include cases like "response does not contain [banned keyword or phrase]", or comparisons between the LLM's response text and some other input data element. 

Each eval is defined with its own folder, containing two files: eval.yaml and prompt.md. The format of these evals as code is very similar to how evals are formatted for other toolkits, such as Arize, so evals previously defined elsewhere in code can be relocated to this framework with minimal difficulty.

**Tasks**

Tasks are the core component of what you are asking the LLM to do. There is a built in class called `PromptTemplateTask` that you can start from, or custom tasks can be easily created as well. All tasks must return a `TaskOutput` type object, which is defined in the framework.

**Models**

This toolkit supports six different model backends (including a "mock" for easy testing), such as AWS Bedrock (regular and Mantle), Ollama, OpenAI, and Anthropic. If there's demand, the number of backends supported may be extended in future. The hookups require tokens and authentication details to be stored in the .env file by the user, and then models are called by specifying their backend and name in the CLI call, such as `bedrock-mantle/google.gemma-4-31b`. Errors fail softly so that if an LLM call has a one-off problem, the entire run is not killed.

**Scoring**

The scoring component takes the evals that have been created and runs them over the LLM responses. This happens immediately after LLM calls, so the process flow creates judge or evaluation results immediately upon receiving the task LLM response. 

Several kinds of evaluation scoring are supported, including LLM-as-a-judge (using the same model access framework as described above), deterministic or programmatic heuristics (such as looking for a key word or the absence of one), or ground-truth comparisons (where the ground truth is provided in the prompt dataset). Latency and token counts (in and out) are also recorded and saved as part of the scoring process, which can be very useful for comparing models in terms of operational considerations.


**Runner**

This is the orchestration component, which organizes and processes the model x prompt case sweeps. It builds the task model client/s (the user can select as many as they want for the tests) and the judge model client initially, then runs each task and the eval scorers. Creating the model clients up front enables fail-fast, so that if any model is unreachable no work is done before discovering this. The output is written to jsonl and is resumable in the case of early stoppage. In addition, the user may add a `--limit` flag which will run only the selected number of cases, which is helpful for smoke testing.

**Results Store**

The runner appends to a results.jsonl file as it runs, and upon conclusion processes this file to a results.csv for a flattened version that can be easily used for analysis. The result is deduped in case a model x prompt case has been run multiple times, taking the most recent model x prompt case pair. 

**Analysis**

This component contains the code to run various statistical tests and mixed effects model, as well as creating the reports, the plots, and the power analysis. The statistical tests are described in detail later in this paper. If the user is not interested in complex analysis, the library can be installed without the optional `analysis` dependencies.

**Shared Spine**

The `GenerationRequest, Response, Usage` object classes contained here provide a decoupled vocabulary, ensuring smooth interaction between the components. There is also a config.py file which loads credentials and provider settings from `.env` for whatever model backend is in use.


## Customization and Extensibility Points

Users have many customization opportunities when using this library. Roughly from lowest to highest lift/complexity:
* Bring prompt cases with detailed inputs
* Choose task LLMs to test from any of the backends supported
* Bring or construct evals
* Choose judge LLM from any of the backends supported
* Create a custom task to test the LLM on
* Add new backend to extend supported LLM offerings
* Create custom scorers, enabling different forms of evals besides LLM-as-a-judge, ground truth, or programmatic checks



## Reuse and Interaction


## Design Decisions/Trade-offs



# Usage / Getting Started

The recommended method for using this library is creating a new repository and adding this library as a dependency. The `init` command in the CLI sets up a basic example with the folder structures that the rest of the tooling expects. This structure will contain boilerplate that will run, and the user can replace it with their custom data, prompts, and evals to customize and get their own project working. 


## Install

Python 3.12+ is required to begin, and `uv` is recommended as the environment management utility.

```uv add "llm-workbench[analysis]"        # or: pip install "llm-workbench[analysis]"```


## Init/Scaffold

```uv run llm-workbench init```

This copies three things into your current directory — files you now own and edit:

```
prompt.txt                              # the prompt template ({input} gets filled per row)
dataset.jsonl                           # a tiny 5-row example test set
evals_kit/example-helpfulness/          # one judge eval (rubric + judge prompt)
    eval.yaml
    prompt.md
```

## Test Run

This test will run out of the box, without model credentials, using mock. Omitting `--judge-model` means there won't be any results scores, but you'll be able to see where they are created. If you do not supply `--out-dir` it will create a folder based on the timestamp.

```
uv run llm-workbench run compare \
    --prompt-template prompt.txt --dataset dataset.jsonl \
    --model mock/echo --eval example-helpfulness --out-dir runs/example
```

## Connect Your LLM Backend

Put credentials in a .env in the project root (loaded automatically; real env vars win over it). Using openai/, anthropic/, or bedrock-mantle/ requires the matching install extra (see Install), but Ollama and plain Bedrock need none. Pick whichever matches what you installed:

```
# Ollama (local, free, no key)      → nothing to set; just have Ollama running
# OpenAI-compatible                 → OPENAI_API_KEY=...   (OPENAI_BASE_URL for others)
# Anthropic                         → ANTHROPIC_API_KEY=...
# Bedrock / Mantle                  → AWS_PROFILE=...  AWS_REGION=...  (SigV4, no key)
```

Confirm it actually works before a full run:

```
ollama pull llama3.1 && ollama pull mistral  # Ollama requires pulling the models first
uv run llm-workbench models ping ollama/llama3.1     # one short call, prints latency + result
```

## Run With Model Backend

If that succeeds, you're ready to run a test case with live models.

This includes two live Ollama models and a judge model, also from Ollama - use your own backend. 

```
uv run llm-workbench run compare \
    --prompt-template prompt.txt --dataset dataset.jsonl \
    --model ollama/llama3.1 --model ollama/mistral \
    --eval example-helpfulness \
    --judge-model ollama/llama3.1 \
    --out-dir runs/example
```

Every model answers every prompt, and the judge scores each answer against the rubric.

## Test Analysis

```uv run llm-workbench analyze runs/example --reference ollama/llama3.1```

This produces plots, the analysis report, and prints analyses to the command line. Add `--verbose` to get a more comprehensive set of analyses at the command line as well. The `--reference` argument indicates which model you'd like to be the comparison item that other models are examined against, but you can omit this and accept the default. 



# Case Study / Worked Example

The following example is a use case where an LLM is asked to extract structured data from a free text passage. We are testing three LLMs on the same task, using custom evals.


After following the initialization and setup instructions shown above, I have the scaffold file structure in place. I am using Ollama for my models, so I am running Ollama locally and have installed a few models.

I checked to ensure that the models I want are available, and the responses look like this.

```
% llm-workbench models ping ollama/llama3.1
✓ ollama/llama3.1  (11.09s, 21 tok)
Hello to you.
```

My initial run using models (retaining the example prompts just for test purposes) looks like this. While the example does use a judge model that is the same as one of the test models, this is NOT appropriate in real working experiments, and the test models should be distinct from the judge.

```
llm-workbench run compare \
--prompt-template prompt.txt --dataset dataset.jsonl \
--model ollama/mistral-small3.2 --model ollama/laguna-xs-2.1:latest \
--eval example-helpfulness \
--judge-model ollama/llama3.1 \
--out-dir runs/example
```

While the prompts are running, terminal output will look like this:

```
Comparing 3 model(s) on 1 eval(s) → runs/example/
models: ollama/llama3.1, ollama/mistral-small3.2, ollama/laguna-xs-2.1:latest   judge: ollama/llama3.1
- ollama/llama3.1  q1
- ollama/llama3.1  q2
- ollama/llama3.1  q3
- ollama/llama3.1  q4
- ollama/llama3.1  q5
· ollama/mistral-small3.2  q1
· ollama/mistral-small3.2  q2
· ollama/mistral-small3.2  q3
· ollama/mistral-small3.2  q4
· ollama/mistral-small3.2  q5
· ollama/laguna-xs-2.1:latest  q1
· ollama/laguna-xs-2.1:latest  q2
· ollama/laguna-xs-2.1:latest  q3
· ollama/laguna-xs-2.1:latest  q4
· ollama/laguna-xs-2.1:latest  q5
```

And the results will render like this in the terminal at the end.

```
Comparison summary                                                                          
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━┳━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┓
┃ model                       ┃ eval        ┃ scorer ┃ n ┃ mean score ┃ pass rate ┃ errors ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━╇━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━┩
│ ollama/laguna-xs-2.1:latest │ Helpfulness │ judge  │ 5 │ 4.8        │           │ 0      │
│ ollama/llama3.1             │ Helpfulness │ judge  │ 5 │ 5          │           │ 0      │
│ ollama/mistral-small3.2     │ Helpfulness │ judge  │ 5 │ 5          │           │ 0      │
└─────────────────────────────┴─────────────┴────────┴───┴────────────┴───────────┴────────┘
Per-model reliability                                                  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ model                       ┃ runs ┃ task errors ┃ mean latency (s) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ mock/echo                   │ 5    │ 0           │ 0.00             │
│ ollama/laguna-xs-2.1:latest │ 5    │ 0           │ 9.60             │
│ ollama/llama3.1             │ 5    │ 0           │ 0.22             │
│ ollama/mistral-small3.2     │ 5    │ 0           │ 10.66            │
└─────────────────────────────┴──────┴─────────────┴──────────────────┘
✓ Done. Results in runs/example/results.jsonl and results.csv
```

After running the analyze command, I get the following at the terminal:

```
Reliability — task completion                          
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ model                ┃ runs ┃ errors ┃ success rate ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━┩
│ echo                 │ 5    │ 0      │ 100%         │
│ laguna-xs-2.1:latest │ 5    │ 0      │ 100%         │
│ llama3.1             │ 5    │ 0      │ 100%         │
│ mistral-small3.2     │ 5    │ 0      │ 100%         │
└──────────────────────┴──────┴────────┴──────────────┘
Reliability = fraction of prompts that produced a usable answer; quality stats below are computed only over successful prompts.

Analysis  models=3  metrics=1  reference=llama3.1  alpha=0.05  margin=±0.2
Design: repeated measures, paired on record. The full statistical battery runs on the quality composite; individual metrics get mean 
tables.
Standardized composite: every metric (graded + binary accuracy) is z-scored within-metric, so all metrics are pooled on a common SD scale. 
Composite means, mixed-model coefficients, Cohen's d and the TOST margin are all in SD units (margin ±0.2 SD).
Overall performance by model (quality       
metrics)                                    
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━┓
┃ model                ┃ mean  ┃ std   ┃ n ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━┩
│ llama3.1  (ref)      │ 5.000 │ 0.000 │ 5 │
│ mistral-small3.2     │ 5.000 │ 0.000 │ 5 │
│ laguna-xs-2.1:latest │ 4.800 │ 0.447 │ 5 │
└──────────────────────┴───────┴───────┴───┘

Mixed Effects Model Results
Mixed model (score ~ C(model)) — diff vs llama3.1                  
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃ model                ┃ coef   ┃ 95% CI         ┃ p      ┃ real? ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ mistral-small3.2     │ -0.000 │ [-1.34, +1.34] │ 1      │ no    │
│ laguna-xs-2.1:latest │ -0.802 │ [-2.14, +0.53] │ 0.2393 │ no    │
└──────────────────────┴────────┴────────────────┴────────┴───────┘
✓ Wrote runs/example/analysis.json and analysis.md
✓ Wrote 2 plot(s): composite_effects.png, composite_violin.png
```

If I used the `--verbose` flag, I would see more results.

## Customizing Cases and Evals

At this point, I'm ready to write my own case prompts and evaluations. Because my task is about extracting formatted data from a text blob, I can create evals that use ground truth and also LLM as a judge.

### Prompt
The default in `prompt.txt` provided is very bare, just 
```
Answer the question concisely.

Question: {input}
Answer:
```

I want to customize this and add some additional elements. 

```
Given a passage of text, extract the date, time, description, and location of the event being described.
Your response should return these values in the following format:

DATE:[date]
TIME:[time]
LOCATION:[location]
DESCRIPTION:[description]

Make the description brief, no more than 50 words. Give the date in MM-DD-YYYY format, and the time in HH:MM AM/PM.

Passage: {passage}
Answer:
```

The prompt cases `dataset.jsonl` will contain the following fields, to allow checking results according to accuracy.

```
{"id": "0005", "passage": "Circle February the 14th on your calendar (2027). Reservations are for seven in the evening at Bellamy's, the little bistro on Chestnut Street downtown — the one with the green awning. It's an anniversary dinner, so dress up a bit.", "expected_date": "02-14-2027",  "expected_time": "07:00 PM",  "expected_location": "Bellamy's on Chestnut Street"}
```

## Evals
I need to design new evals that will make sense with these tasks. 

First, I want to simply compare the time and date to make sure the model got these right. Second, I want to make sure the description is broadly correct, and I'll use an llm-as-a-judge for that. To set these up, I create new folders in `evals_kit`: `date_correct`, `time_correct`, and `description_accuracy`. (I deleted the example for helpfulness after using it for a template.)

Each eval folder takes one `eval.yaml` and one (optional) `prompt.md`. Eval contains specifics and prompt is, naturally, the prompt the LLM judge receives.

For coverage, I'm making Date Correct a "contains" eval and Description a "judge" eval so the reader can see multiple possibilities.

### Contains
For Date Correct, I'm making it a very simple check- does the output from the LLM contain the value I've listed as the expected date? I'm being exact, so if the LLM did not follow my formatting requirements, and gives the date in the wrong format, it will fail. This is something I have to decide on as the user, but fuzzy matching is also possible. For example, I could ask the LLM judge to compare the desired output with what the task LLM rendered.

``` 
eval.yaml

name: Date Correctness
description: >-
  Judges whether the response contains the expected date output.

scorers:
  - type: contains
    expected_field: expected_date
```

### Judge
For judge, we are using an LLM as a judge, potentially including extra facts from the dataset, but not necessarily. For this case, I'm going to have an LLM assess whether the event description matches roughly what the prompt indicated. I don't have a ground truth in place, just the original passage. Basically, I'm having a second LLM do an additional pass over the question, getting a review of the first LLM's effort.

This calls for both the `eval.yaml` and `prompt.md`.

```
eval.yaml

name: Description Accuracy
description: >-
  Judges whether the extracted event description matches what the passage actually describes.
direction: MAXIMIZE
labels:
  "Accurately captures what the event is, with no errors.": 5
  "Captures the event with minor omissions or vagueness.": 4
  "Roughly right but generic or missing a notable aspect.": 3
  "Partially wrong, or conflates the event with irrelevant detail.": 2
  "Wrong event, or no usable description.": 1
prompt_file: prompt.md
```

```
prompt.md

You are checking how well a model's extracted event DESCRIPTION matches the event actually described in a passage.

The passage below describes a single event. The model was asked to extract the date, time, location, and a brief description of that event. Judge ONLY the description: does it accurately convey what the event is (its occasion or purpose), allowing for reasonable paraphrase? Ignore the date, time, and location fields — those are scored separately. A description can be correct even if worded differently from the passage, as long as it captures the right event.

<passage>
{passage}
</passage>

<model_output>
{output}
</model_output>
```

### Custom Evals
To use custom pattern lookups requires a little more effort, in the form of a custom scorer. You can use this if your LLM output contains complex formatting, where you want to look for a particular value in context, or you can use it to look for several values at once. Build out means writing a couple of extra files. This is described separately in this paper.



## Run

Now I'm ready to run the experiment, and I'll return to the format from earlier. I've changed the output folder so it doesn't combine with the smoke test versions.

```
llm-workbench run compare \
--prompt-template prompt.txt --dataset dataset.jsonl \
--model ollama/mistral-small3.2 --model ollama/laguna-xs-2.1:latest \
--judge-model ollama/llama3.1 \
--out-dir runs/example_2
```



## Customizing Evaluations

An LLM judge isn't always the best choice for a use case. When a field has an objectively correct answer stored in your dataset, you can score it deterministically. This section gives an overview of simple "contains" judging and also LLM-as-a-judge options.


### Step 1 — Confirm the gold value is in your dataset
Each record must carry the correct answer. Our dataset already includes it:

```
{"id": "0003", "passage": "The team offsite is happening on Thursday, April 16th, 2026. We'll start at 9:30 in the morning ...", "expected_time": "09:30 AM"}
```

expected_time is the value the eval will grade against.

### Step 2 — Write a task that exposes the parsed answer
The default prompt-template task returns only the raw model output. Ground-truth scoring needs the extracted time specifically, so we write a small custom task that runs the model and places the parsed value in aux — a dictionary the eval can read by name. This output should come as the provided TaskOutput class object.

```
# event_task.py
import re
from pathlib import Path

from llm_workbench.object_types import GenerationRequest
from llm_workbench.tasks.base import TaskOutput
from llm_workbench.evals.template import fill_template


class ExtractionTask:
    name = "event-extraction"

    def __init__(self):
        self.template = Path("prompt.txt").read_text()

    def run(self, record, client):
        prompt = fill_template(self.template, record)
        response = client.generate(GenerationRequest(prompt=prompt, max_tokens=256))
        text = response.text or ""

        # Pull the value from the "TIME:" line into aux so the eval can grade it.
        match = re.search(r"TIME:\s*(.+)", text)
        parsed_time = match.group(1).strip() if match else ""

        return TaskOutput(
            output=text,
            prompt=prompt,
            response=response,
            aux={"time": parsed_time},
        )

```

The key line is aux={"time": parsed_time}: whatever the task puts in aux becomes available to the eval by that name. If you create multiple ground truth evals, you can add the fields to this class and reuse it instead of creating more. Also, notice that the `prompt.txt` is called in here- if you change which prompt you're using, make sure to update it here.

### Step 3 — Create the eval
This follows the previously laid out structure, where an eval folder contains the yaml and the prompt. `eval.yaml` declares the ground-truth comparison:

```
# evals_kit/time_correct/eval.yaml
name: Time Correctness
description: The extracted time matches the expected time.
direction: MAXIMIZE
labels:
  correct: 1
  incorrect: 0
prompt: "Fallback only — unused when expected_time is present."
ground_truth:
  expected_field: expected_time   # the gold value in each dataset record
  aux_field: time                 # the value the task placed in aux
```

The ground_truth block is what turns this into a deterministic accuracy check: for every record that has expected_time, the eval compares it to aux["time"] and skips the LLM judge entirely. We use an inline prompt placeholder because there is no prompt required for a ground truth test.


### Step 4 — Run the comparison
Because the extraction logic now lives in the task, we pass --task instead of --prompt-template:

```
uv run llm-workbench run compare \
  --task event_task:ExtractionTask \
  --dataset dataset.jsonl \
  --model ollama/llama3.1 --model ollama/mistral-small3.2 \
  --judge-model ollama/llama3.1 \
  --out-dir runs/experiment
```

Two flags deserve a note. First, --task and --prompt-template are mutually exclusive — the custom task builds its own prompt (here by reading prompt.txt). Second, --judge-model is required even though the judge is bypassed: the ground-truth logic is part of the judge scorer, which is only activated when a judge model is supplied. It will not actually be called for records that carry the gold value, but the flag must be present.

The eval emits a binary metric — 1.0 when the parsed time matches, 0.0 otherwise. By default it is standardized and pooled into the quality composite alongside the judged metrics; use --raw for a graded-only composite, or --exclude-metric to report it as a standalone accuracy track.



# Statistical Methodology

The premise of this library is that LLM performance can be effectively tested with a paired prompt comparison strategy. This means that every LLM is fed the same pool of cases, and the differences in their behavior across the spectrum of the designated evals are what we intend to measure. As a result, we are able to eliminate variance due to differences in prompt difficulty, and can attribute the differences to the actual LLM performance.

How the evaluations are designed is very important to achieving a rigorous measure of quality in this way. Evals should have thoughtfully designed spectra of performance, and should not have significant overlap in the scope of what they measure. Our goal is to produce measurement validity over the broad area of "LLM quality" on the specific task in question. This will have many attributes in most cases, but combining them together to create the composite of "quality" with high construct validity is our objective.

## Designing the Construct through Evaluations
Each evaluation you write produces scores on its own scale — a graded rubric might run 1–5, while a ground-truth check is binary. To combine them into a single measure of quality without letting the widest-ranged metric dominate, we standardize each metric independently: every score is re-expressed as a z-score, i.e. how many standard deviations it sits above or below that metric's own mean across the run. 

On this common scale, a single standard deviation shift means the same thing whether it came from the rubric or the accuracy check. The composite quality score for a given model on a given prompt is then simply the mean of its standardized metric scores. Metrics that show no variation across the run — every model scoring identically — are dropped rather than counted as zeros: a metric that cannot distinguish any of the models carries no signal about quality, and including it would only dilute the composite. 

At this time, the framework weights all metrics equally in constructing the composite, but adjusting weights may be supported in the future. If a user wants to construct the composite excluding a metric, to test the effect on the overall analysis, `--exclude-metric` is an available parameter on the analysis call. 

## Statistical Tests Included

**Do the models differ?**

When there are three or more models with results generated, we begin by examining whether there is a difference detectable. For each prompt we rank the models best-to-worst, and the Friedman test asks whether one model tends to rank consistently higher than the others across the whole prompt set. It is the paired, rank-based counterpart to a one-way ANOVA, and because it operates on ranks it makes no assumption that the scores are normally distributed. 

Additionally the Kendall's W is calculated, giving an effect size from 0 (models ranked essentially at random relative to one another) to 1 (every prompt produces the identical ranking), which tells us how consistent the ordering is, not just whether it is detectable. 

**What are the differences?**

The next step does unordered paired difference analysis between each pair of models. The Wilcoxon signed-rank test is the paired, non-parametric analog of a paired t-test, which looks at whether the per-prompt differences between two models lean consistently in one direction. 

With three or more models this serves as the post-hoc following Friedman; with exactly two models it is the primary test. Because comparing many pairs inflates the chance of a false positive, we apply the Holm correction to the p-values, which controls the family-wise error rate while remaining more powerful than the more familiar Bonferroni adjustment. For each pair we report the higher-scoring model, the corrected p-value, and whether the difference clears significance.

**The Size of Difference**

Up to now we have calculated whether the models differ, and looked at them in a round-robin pairing fashion. However, a mixed-effects model is the next step to tell us the size and direction of the difference. This model fits score ~ model, with a random intercept for each prompt. The random intercept is what encodes the pairing: it lets every prompt sit at its own baseline difficulty and asks whether a given model scores consistently above or below that prompt's baseline. 

The output is, for each model, its estimated difference from the reference model, with a confidence interval and a p-value. When more than one metric feeds the composite, the model also includes a term for the metric so that differing metric baselines do not distort the estimates. This is the one test we compute in every run, even in the compact output mode, because the per-model differences-from-reference are the headline result most users are after.

**Standardized Effect Size**

For each pair of models we also report Cohen's d, which is the mean of the paired differences divided by the standard deviation of those differences. It is important that the denominator is the standard deviation of the differences, not of the raw scores — this is the correct form for a paired design, and it expresses the typical gap between two models relative to how much that gap wobbles from prompt to prompt. We label the magnitude on the conventional scale (negligible, small, medium, large at 0.2, 0.5, 0.8). Because Cohen's d isolates effect size from sample size, it answers "how much" independently of whether the difference reached significance.

**If No Effect, Are Models the Same?**

The TOST equivalence test is most useful when we find no statistically significant difference in the other tests. A finding of no difference in the mixed effects model tells us that the difference could be due to chance or insufficient data, and it doesn't necessarily prove that the models are statistically equivalent (meaning, it doesn't matter which one we choose for a task). To make an affirmative claim of equivalence we use TOST (two one-sided tests), which inverts the usual logic: the null hypothesis becomes "the models differ by more than a margin we care about," and rejecting it lets us conclude that any difference is small enough to be practically irrelevant. Each model is tested against the reference within a margin the user sets (in standardized units, since the composite is standardized). 

**Other Tests**

Two further tests are also included. A bootstrap confidence interval on the mean paired difference (resampling the prompts many times and recomputing) provides a distribution-free cross-check on the parametric intervals, making no assumptions about the shape of the data. 
We also report the skewness of each model's score distribution, because rubric scores often pile up near the top of the scale. If the LLMs being tested all perform rather well, we might see scores highly skewed, and knowing the degree of that skew helps us know what effect it has on our results. This is also one reason why the Wilcoxon signed-rank test is valuable, as it is non-parametric.



Not all tests are run with every analysis flow, for convenience. Use `--verbose` to ensure every test is produced and saved in the report. Also, if one or more LLMs fail to produce a successful response to a prompt case, that prompt case will be excluded from the Friedman/Wilcoxon tests, as these are rank based and require a response from every model in testing to be calculated.

# Related Work / Comparison

Comparison to alternative frameworks; where llm-workbench fits

# Limitations & Roadmap

Current constraints and known gaps
Planned direction

# Community & Contributing

License, governance, how to contribute, support channels

# Conclusion

Appendices / References

API reference pointers, extended examples, citations